# R2-02 · SQL para *features* — Profundización (opcional) 🔬

**Formación Pública — Rama R2 · Científico de Datos · Notebook de profundización**

Opcional y de **transferencia**: el mismo oficio de fabricar *features* con SQL sobre otro dato
público — los **1.601 sismos de magnitud ≥ 4,5** registrados en Chile entre 2020 y 2025 (catálogo
del Servicio Geológico de EE.UU., USGS). Cambia la entidad (la **zona** del país), aparecen las
banderas 0/1, el rezago entre **fechas** con `julianday`, y una herramienta nueva: la **CTE**
(`WITH`) para agregar una *feature* de ventana.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import os, urllib.request, sqlite3
import pandas as pd, numpy as np

CSV = "sismos.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/sismos.csv", CSV)
sis = pd.read_csv(CSV)    # sismos M>=4,5 en Chile 2020–2025 (catálogo USGS)

con = sqlite3.connect(":memory:")
sis.to_sql("sismos", con, index=False)

def correr(sql):
    return pd.read_sql_query(sql, con)

print("Tabla 'sismos':", len(sis), "eventos;", "zonas:", sorted(sis["zona"].unique()))
correr("SELECT * FROM sismos LIMIT 5")

## 1. *Features* por zona y año

Mismo `GROUP BY` de la lección, otra entidad: la **zona** del país y el **año**. `strftime('%Y', fecha)` extrae el año.

### ✍️ Ejercicio 1 — Resumen sísmico por zona y año

Una fila por `zona` y `anio` (entero) con `n_sismos`, `magnitud_promedio` (2 decimales) y
`magnitud_maxima`. Ordena por `zona` y `anio`.

In [ ]:
consulta_1 = """
SELECT zona,
       CAST(strftime('%Y', fecha) AS INTEGER) AS anio,
       COUNT(*)                 AS n_sismos,
       ROUND(AVG(magnitud), 2)  AS magnitud_promedio,
       MAX(magnitud)            AS magnitud_maxima
FROM sismos
GROUP BY zona, anio
ORDER BY zona, anio
"""
correr(consulta_1)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    r = correr(consulta_1)
    assert {"zona", "anio", "n_sismos", "magnitud_promedio", "magnitud_maxima"}.issubset(r.columns)
    assert len(r) == sis.groupby([sis["zona"], sis["fecha"].str[:4]]).ngroups
    assert int(r["n_sismos"].sum()) == len(sis)
    assert r["magnitud_maxima"].max() == sis["magnitud"].max()
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "GROUP BY zona, anio con CAST(strftime('%Y', fecha) AS INTEGER) AS anio.")
    print("   Detalle:", e)

## 2. Etiquetas con `CASE WHEN`: profundidad y sismo fuerte

Dos *features* condicionales por evento: el tipo de profundidad (superficial < 70 km, intermedio 70–300 km, profundo > 300 km) y una **bandera 0/1** `es_fuerte` para M ≥ 6.

### ✍️ Ejercicio 2 — Clasifica cada sismo

Devuelve `fecha`, `zona`, `profundidad_km`, `magnitud`, `profundidad_tipo` y `es_fuerte`.

In [ ]:
consulta_2 = """
SELECT fecha, zona, profundidad_km, magnitud,
       CASE
           WHEN profundidad_km < 70   THEN 'superficial'
           WHEN profundidad_km <= 300 THEN 'intermedio'
           ELSE 'profundo'
       END AS profundidad_tipo,
       CASE WHEN magnitud >= 6 THEN 1 ELSE 0 END AS es_fuerte
FROM sismos
"""
r2 = correr(consulta_2)
print(r2["profundidad_tipo"].value_counts(), "\nfuertes:", r2["es_fuerte"].sum())

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    r = correr(consulta_2)
    assert {"profundidad_tipo", "es_fuerte"}.issubset(r.columns)
    c = r["profundidad_tipo"].value_counts()
    assert c.get("superficial", 0) == int((sis["profundidad_km"] < 70).sum())
    assert c.get("intermedio", 0) == int(((sis["profundidad_km"] >= 70) & (sis["profundidad_km"] <= 300)).sum())
    assert int(r["es_fuerte"].sum()) == int((sis["magnitud"] >= 6).sum())
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Dos CASE: el de profundidad (de menor a mayor umbral con <) y CASE WHEN magnitud >= 6 THEN 1 ELSE 0 END AS es_fuerte.')
    print("   Detalle:", e)

## 3. Rezago temporal: días desde el sismo anterior en la zona

`LAG(fecha)` trae la fecha del sismo anterior **de la misma zona**; `julianday()` convierte fechas en números de días, así que la resta da los días transcurridos. Es una *feature* **punto-en-el-tiempo** correcta: en el momento de cada sismo, el anterior ya ocurrió.

> 🚨 Lo contrario sería *leakage*: «días **hasta el próximo** sismo» (`LEAD`) no se conoce cuando ocurre el evento y jamás debe entrar a un modelo predictivo.

### ✍️ Ejercicio 3 — Calcula `dias_desde_anterior`

Devuelve `fecha`, `zona`, `magnitud` y `dias_desde_anterior`, ordenado por `zona` y `fecha`.

In [ ]:
consulta_3 = """
SELECT fecha, zona, magnitud,
       julianday(fecha) - julianday(LAG(fecha) OVER (PARTITION BY zona ORDER BY fecha))
           AS dias_desde_anterior
FROM sismos
ORDER BY zona, fecha
"""
r3 = correr(consulta_3)
r3.groupby("zona")["dias_desde_anterior"].describe().round(1)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    r = correr(consulta_3)
    assert "dias_desde_anterior" in r.columns
    assert len(r) == len(sis)
    assert int(r["dias_desde_anterior"].isna().sum()) == sis["zona"].nunique(), "solo el primer sismo de cada zona debe quedar NULL"
    assert r["dias_desde_anterior"].min() >= 0, "¿ordenaste por fecha dentro del OVER?"
    esperado = sis.assign(f=pd.to_datetime(sis["fecha"])).sort_values(["zona", "f"]).groupby("zona")["f"].diff().dt.days
    assert abs(r["dias_desde_anterior"].sum() - esperado.sum()) < 1e-6
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'julianday(fecha) - julianday(LAG(fecha) OVER (PARTITION BY zona ORDER BY fecha)).')
    print("   Detalle:", e)

## 4. La tabla analítica por zona (con una CTE)

Para agregar una *feature* de ventana hay que calcularla **primero** y agrupar **después**. Una **CTE** (`WITH t AS (...)`) es una tabla temporal con nombre: calculas la ventana en `t` y agrupas sobre `t`.

### ✍️ Ejercicio 4 — Una fila por zona

Una fila por `zona` con `n_sismos`, `magnitud_promedio` (2 decimales), `n_fuertes` (M ≥ 6) y
`dias_entre_sismos` (promedio de `dias_desde_anterior`, 1 decimal). Ordena por `n_sismos` desc.

In [ ]:
consulta_4 = """
WITH t AS (
    SELECT zona, magnitud,
           julianday(fecha) - julianday(LAG(fecha) OVER (PARTITION BY zona ORDER BY fecha))
               AS dias_desde_anterior
    FROM sismos
)
SELECT zona,
       COUNT(*)                                         AS n_sismos,
       ROUND(AVG(magnitud), 2)                          AS magnitud_promedio,
       SUM(CASE WHEN magnitud >= 6 THEN 1 ELSE 0 END)   AS n_fuertes,
       ROUND(AVG(dias_desde_anterior), 1)               AS dias_entre_sismos
FROM t
GROUP BY zona
ORDER BY n_sismos DESC
"""
correr(consulta_4)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    r = correr(consulta_4)
    assert {"zona", "n_sismos", "magnitud_promedio", "n_fuertes", "dias_entre_sismos"}.issubset(r.columns)
    assert len(r) == sis["zona"].nunique()
    assert int(r["n_sismos"].sum()) == len(sis)
    assert int(r["n_fuertes"].sum()) == int((sis["magnitud"] >= 6).sum())
    assert r.set_index("zona")["dias_entre_sismos"].idxmax() == "austral", "la zona con menos sismos debería tener los intervalos más largos"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'WITH t AS (SELECT zona, magnitud, <la resta de julianday con LAG> AS dias_desde_anterior FROM sismos) SELECT zona, COUNT(*), ..., AVG(dias_desde_anterior) FROM t GROUP BY zona.')
    print("   Detalle:", e)

---

*Fin de la profundización de R2-02.* Fuente: USGS Earthquake Catalog ([earthquake.usgs.gov](https://earthquake.usgs.gov)).